# QianfanOCR - 文档解析（Document Parsing）

将文档图片（PDF 中提取的一页或多页）高保真地转换为 Markdown 格式，支持文本、公式（LaTeX）、表格（HTML）、图片占位符等元素的结构化输出。

> **提示**：为体验更好的解析效果，或针对**报纸**这类复杂版面场景，建议将 `min_dynamic_patch` 设为 **8**，`max_dynamic_patch` 设为 **24**。

## 1. 环境准备

In [ ]:
import base64
import requests

## 2. 配置参数

请将 `API_URL` 和 `API_KEY` 替换为实际的服务地址和密钥。

In [ ]:
API_URL = "http://your-api-endpoint/v1/chat/completions"
API_KEY = "your-api-key"

# 模型超参
MODEL_NAME = "ocrfp8"
MAX_TOKENS = 4096
TEMPERATURE = 0.3
SKIP_SPECIAL_TOKENS = False

# 复杂版面场景超参（报纸、杂志等多栏密集排版）
MAX_TOKENS_COMPLEX = 16384
MIN_DYNAMIC_PATCH = 8
MAX_DYNAMIC_PATCH = 24

## 3. 读取图片

将待解析的文档图片编码为 base64 格式。

In [ ]:
image_path = "../images/page_001.jpg"

with open(image_path, "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")

print(f"图片已加载: {image_path}")

## 4. 调用文档解析 API

In [ ]:
DOC_PARSING_PROMPT = """You are an AI assistant specialized in converting document images (one or multiple pages extracted from a PDF) into Markdown with high fidelity.

Your task is to accurately convert all visible content from the images into Markdown, strictly following the rules below. Do not add explanations, comments, or inferred content.

1. Pages:
- The input may contain one or multiple page images.
- Preserve the exact page order as provided.
- If there are multiple pages, separate pages using the marker:
  --- Page N ---
  (N starts from 1)
- If there is only one page, do NOT output any page separator.

2. Text Recognition:
- Accurately convert all visible text.
- No guessing, inference, paraphrasing, or correction.
- Preserve the original document structure, including headings, paragraphs, lists, captions, and footnotes.
- Completely REMOVE all header and footer text. Do not output page numbers, running titles, or repeated marginal content.

3. Reading Order:
- Follow a top-to-bottom, left-to-right reading order.
- For multi-column layouts, fully read the left column before the right column.
- Do not reorder content for semantic or logical clarity.

4. Mathematical Formulas:
- Convert all mathematical expressions to LaTeX.
- Inline formulas must use $...$.
- Display (block) formulas must use:
  $$
  ...
  $$
- Preserve symbols, spacing, and structure exactly.
- Do not invent, simplify, normalize, or correct formulas.

5. Tables:
- Convert all tables to HTML format.
- Wrap each table with <table> and </table>.
- Preserve row and column order, merged cells (rowspan, colspan), and empty cells.
- Do not restructure or reinterpret tables.

6. Images:
- Do NOT describe image content.
- Preserve images using the exact format:
  ![label](<box>[[x1, y1, x2, y2]]</box>)
- Allowed labels: image, chart, seal.
- Completely REMOVE all header_image and footer_image elements.
- Do not introduce new labels.
- Do not remove or merge remaining image elements.

7. Unreadable or Missing Content:
- If text, symbols, or table cells are unreadable, preserve their position and leave the content empty.
- Do not guess or fill in missing information.

8. Output Requirements:
- Output Markdown only.
- Preserve original layout, spacing, and structure as closely as possible.
- Ensure clear separation between elements using line breaks.
- Do not include any explanations, metadata, or comments."""

In [ ]:
payload = {
    "model": MODEL_NAME,
    "max_tokens": MAX_TOKENS,
    "temperature": TEMPERATURE,
    "mm_processor_kwargs": {
        "skip_special_tokens": SKIP_SPECIAL_TOKENS
    },
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpg;base64,{image_base64}"
                    }
                },
                {
                    "type": "text",
                    "text": DOC_PARSING_PROMPT
                }
            ]
        }
    ]
}

response = requests.post(
    API_URL,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    },
    json=payload
)

result = response.json()["choices"][0]["message"]["content"]
print(result)

## 5. 针对复杂版面的参数调优

对于报纸、杂志等多栏密集排版场景，建议添加额外参数以获得更好的解析效果。

In [ ]:
payload_complex = {
    "model": MODEL_NAME,
    "max_tokens": MAX_TOKENS_COMPLEX,
    "temperature": TEMPERATURE,
    "mm_processor_kwargs": {
        "skip_special_tokens": SKIP_SPECIAL_TOKENS,
        "min_dynamic_patch": MIN_DYNAMIC_PATCH,
        "max_dynamic_patch": MAX_DYNAMIC_PATCH,
    },
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpg;base64,{image_base64}"
                    }
                },
                {
                    "type": "text",
                    "text": DOC_PARSING_PROMPT
                }
            ]
        }
    ]
}

response_complex = requests.post(
    API_URL,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    },
    json=payload_complex
)

result_complex = response_complex.json()["choices"][0]["message"]["content"]
print(result_complex)